In [ ]:
# Redémarrage automatique du notebook
%reset -f
import sys
import os

# Ajouter le dossier parent (la racine du projet) au PYTHONPATH
sys.path.append(os.path.abspath(".."))

In [ ]:
# importation des librairies
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.feature_selection import RFE, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from textwrap import fill

from src.data_processing import optimize_memory, fit_preprocessor_on_train, transform_with_train_preprocessor
from src.features import rfe_logreg

In [ ]:
# Load the dataset
data = pd.read_excel('../data/raw/app_data.xlsx', sheet_name='All cases')
print(data['Age'].min(), data['Age'].max())

data.columns = data.columns.str.strip().str.replace(r'\s+', '_', regex=True)
# Display the first few rows of the dataset
#print(data.head())

In [ ]:
#Shape of the dataset
print("Shape of the dataset:", data.shape)
# Data types of each column
print("\nData types of each column:\n", data.dtypes)
# Check for missing values
print("\nCount des valeurs manquantes par colonne :")
print(data.isnull().sum())
print("\nnombre de valeurs uniques par colonne :")
print(data.nunique())

# Affichage des valeurs manquantes en 6 graphiques (58 colonnes ÷ 6)
missing = data.isnull().sum()
cols = missing.index.tolist()
n_graphs = 6
chunk_size = len(cols) // n_graphs + (1 if len(cols) % n_graphs else 0)

fig, axes = plt.subplots(n_graphs, 2, figsize=(14, 4 * n_graphs))
axes = axes.flatten()
for i, ax in enumerate(axes):
    chunk = cols[i * chunk_size:(i + 1) * chunk_size]
    if not chunk:
        ax.set_visible(False)
        continue
    vals = missing[chunk]
    ax.bar(vals.index, vals.values, color="steelblue")
    ax.axhline(y=data.shape[0], color="red", linestyle="--", label=f"Total lignes ({data.shape[0]})")
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=90, fontsize=8)
    ax.set_title(f"Valeurs manquantes — colonnes {i*chunk_size+1} à {min((i+1)*chunk_size, len(cols))}")
    ax.set_ylabel("Nb valeurs manquantes")
    ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
for col in data.columns:
    if data[col].dtype == 'float64':
        data[col] = data[col].astype(float)

In [ ]:
# Renommer la colonne cible et préparer le dataset de modélisation
target_col = "Diagnosis"
columns_to_exclude = []

df_model = data.copy()
df_model = df_model.rename(columns={"Diagnosis_no appendicitis": target_col})

target_series = pd.to_numeric(df_model[target_col], errors="coerce")
if target_series.isna().any():
    normalized_target = (
        df_model[target_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[_\s]+", " ", regex=True)
    )
    fallback_mapping = {
        "0": 0,
        "1": 1,
        "false": 0,
        "true": 1,
        "no": 1,
        "yes": 0,
        "no appendicitis": 1,
        "appendicitis": 0,
    }
    target_series = normalized_target.map(fallback_mapping)

valid_target_mask = target_series.notna()
if not valid_target_mask.all():
    print(f"Lignes retirées car cible manquante ou illisible : {(~valid_target_mask).sum()}")
    df_model = df_model.loc[valid_target_mask].copy()
    target_series = target_series.loc[valid_target_mask]

df_model[target_col] = 1 - target_series.astype(int)

if columns_to_exclude:
    df_model = df_model.drop(columns=columns_to_exclude, errors="ignore")

df_model = optimize_memory(df_model)

print("Colonne cible préparée : 1 = appendicite, 0 = pas d'appendicite")
print(f"Colonnes explicitement exclues avant split : {columns_to_exclude}")
print(f"Shape du dataset de modélisation : {df_model.shape}")

In [ ]:
# Split avant preprocessing pour éviter le data leakage
y = df_model[target_col].astype(int)
X = df_model.drop(columns=[target_col])

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
 )

X_train_preprocessed, meta = fit_preprocessor_on_train(
    X_train_raw,
    missing_threshold=0.999,
    iqr_factor=1.5,
    drop_first=True
 )
X_test_preprocessed = transform_with_train_preprocessor(X_test_raw, meta)

train_processed = X_train_preprocessed.copy()
train_processed[target_col] = y_train.to_numpy()

test_processed = X_test_preprocessed.copy()
test_processed[target_col] = y_test.to_numpy()

print(f"Train brut : {X_train_raw.shape} | Test brut : {X_test_raw.shape}")
print(f"Train prétraité : {X_train_preprocessed.shape} | Test prétraité : {X_test_preprocessed.shape}")
print(
    "Colonnes supprimées sur la base du train (NaN > seuil) :",
    meta["columns_dropped_for_missingness"]
 )

In [ ]:
print("Shape du train prétraité :", train_processed.shape)
print("Shape du test prétraité :", test_processed.shape)
print("\nValeurs manquantes restantes dans le train prétraité :", train_processed.isnull().sum().sum())
print("Valeurs manquantes restantes dans le test prétraité :", test_processed.isnull().sum().sum())
print("\nLes colonnes catégorielles ajustées sur le train sont :")
print(meta["categorical_columns"], "De taille :", len(meta["categorical_columns"]))
print("\nLes colonnes encodées retenues depuis le train sont :")
print(meta["encoded_columns"], "De taille :", len(meta["encoded_columns"]))

In [ ]:
# Répartition de la cible après le split
print(y_train.value_counts().sort_index())
print(y_train.value_counts(normalize=True).sort_index().round(3) * 100)

target_distribution = pd.DataFrame({
    "train": y_train.value_counts(normalize=True).sort_index(),
    "test": y_test.value_counts(normalize=True).sort_index(),
}).fillna(0)

print("\nRépartition relative train/test (%) :")
print((target_distribution * 100).round(2))

ax = y_train.value_counts().sort_index().plot(
    kind="bar",
    color=["#4C78A8", "#F58518"],
    title="Répartition de la cible dans le train"
 )
ax.set_xticklabels(["Pas d'appendicite", "Appendicite"], rotation=0)
ax.set_xlabel("Classe")
ax.set_ylabel("Nombre d'échantillons")
plt.show()

In [ ]:
# EDA et sélection de features sur le train uniquement
y_eda = y_train.reset_index(drop=True)
X_eda = X_train_preprocessed.reset_index(drop=True)

num_cols = X_eda.select_dtypes(include=[np.number]).columns
corrs = X_eda[num_cols].corrwith(y_eda).abs().sort_values(ascending=False)
corrs.head(15)

In [ ]:
# Heatmap de corrélation entre features numériques (après traitement et encodage OHE)
num_cols_all = X_eda.select_dtypes(include=[float]).columns
corr_num = X_eda[num_cols_all].corr()

plt.figure(figsize=(16, 14))
sns.heatmap(
    corr_num,
    cmap="coolwarm",
    center=0,
    linewidths=0.3,
    annot=True,
    fmt=".2f",
    annot_kws={"size": 8},
    square=False,
    cbar_kws={"shrink": 0.8}
)
plt.title("Matrice de corrélation — Features numériques (après encodage)", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# MI calculée sur le train prétraité
mi = mutual_info_classif(X_eda[num_cols], y_eda, random_state=42)
mi_series = pd.Series(mi, index=num_cols).sort_values(ascending=False)
mi_series.head(15)

In [ ]:
corr_matrix = X_eda[num_cols].corr().abs()
threshold = 0.75

print("Matrice de corrélation (extrait) :")
print(corr_matrix.iloc[:5, :5])

upper = np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
high_pairs = [
    (corr_matrix.index[i], corr_matrix.columns[j], corr_matrix.values[i, j])
    for i in range(corr_matrix.shape[0])
    for j in range(corr_matrix.shape[1])
    if upper[i, j] and corr_matrix.values[i, j] > threshold
]

print(f"Nombre de paires avec corrélation > {threshold}: {len(high_pairs)}")
pd.DataFrame(high_pairs, columns=["feature_1", "feature_2", "correlation"]).sort_values(
    by="correlation", ascending=False
).head(10)

if high_pairs:
    n_pairs = len(high_pairs)
    n_cols_plot = 2
    n_rows_plot = (n_pairs + n_cols_plot - 1) // n_cols_plot
    fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(12, 4.5 * n_rows_plot))
    axes = np.array(axes).reshape(-1)

    for idx, (feature_1, feature_2, corr_value) in enumerate(high_pairs):
        sns.scatterplot(
            data=X_eda,
            x=feature_1,
            y=feature_2,
            ax=axes[idx],
            alpha=0.7,
            s=35
        )
        axes[idx].set_title(f"{feature_1} vs {feature_2} (r={corr_value:.2f})")

    for idx in range(len(high_pairs), len(axes)):
        axes[idx].set_visible(False)

    plt.suptitle(f"Scatter plots des variables avec corrélation > {threshold}", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()
else:
    print(f"Aucune paire de variables ne dépasse le seuil de corrélation de {threshold}.")

# Règle simple : si deux features > 0.75 corrélées, n’en garder qu’une (selon MI ou pertinence clinique).


In [ ]:
# Répartition des colonnes catégorielles selon la cible sur le train uniquement
cat_cols = [c for c in meta["categorical_columns"] if c in X_train_raw.columns]

if not cat_cols:
    print("Aucune colonne catégorielle à visualiser après préparation du train.")
else:
    plot_data = X_train_raw[cat_cols].copy()
    plot_data[target_col] = y_train.map({1: "Appendicite", 0: "Pas appendicite"})

    plots_per_figure = 4
    n_cols_plot = 2
    n_rows_plot = 2
    max_categories = 6

    for start_idx in range(0, len(cat_cols), plots_per_figure):
        batch_cols = cat_cols[start_idx:start_idx + plots_per_figure]
        fig, axes = plt.subplots(
            n_rows_plot,
            n_cols_plot,
            figsize=(16, 11),
            constrained_layout=True
        )
        axes = np.array(axes).reshape(-1)

        for ax_idx, col in enumerate(batch_cols):
            ax = axes[ax_idx]
            current = plot_data[[col, target_col]].copy()
            current[col] = current[col].astype("object")
            current[col] = current[col].where(current[col].notna(), "Missing").astype(str)

            value_counts = current[col].value_counts()
            if value_counts.shape[0] > max_categories:
                top_categories = value_counts.head(max_categories - 1).index
                current[col] = current[col].where(current[col].isin(top_categories), "Autres")
                value_counts = current[col].value_counts()

            order = value_counts.index.tolist()
            max_label_length = max(len(label) for label in order)
            use_horizontal = len(order) > 4 or max_label_length > 18

            if use_horizontal:
                wrapped_map = {category: fill(category, 20) for category in order}
                current[col] = current[col].map(wrapped_map)
                wrapped_order = [wrapped_map[category] for category in order]
                sns.countplot(
                    data=current,
                    y=col,
                    hue=target_col,
                    order=wrapped_order,
                    ax=ax,
                    palette="Set2"
                )
                ax.set_xlabel("Count")
                ax.set_ylabel("")
                ax.tick_params(axis="y", labelsize=8)
            else:
                sns.countplot(
                    data=current,
                    x=col,
                    hue=target_col,
                    order=order,
                    ax=ax,
                    palette="Set2"
                )
                ax.set_xlabel("")
                plt.setp(ax.get_xticklabels(), rotation=20, ha="right", fontsize=8)

            ax.set_title(fill(col, 28), fontsize=11)

            if ax_idx == 0:
                ax.legend(title=target_col, fontsize=8, title_fontsize=9, loc="upper right")
            else:
                legend = ax.get_legend()
                if legend is not None:
                    legend.remove()

        for empty_ax in axes[len(batch_cols):]:
            empty_ax.set_visible(False)

        fig.suptitle(
            f"Répartition des features catégorielles sur le train ({start_idx + 1} à {start_idx + len(batch_cols)})",
            fontsize=14,
            y=1.02
        )
        plt.show()
        plt.close(fig)

In [ ]:
# Sélection finale des features sur le train prétraité
rfe_input = X_train_preprocessed.copy()
rfe_input[target_col] = y_train.to_numpy()

top_corr = rfe_logreg(rfe_input, target_col, n_features=15, step=1)
print("Top 15 features selon RFE + LogReg sur le train :")
print(top_corr)

In [ ]:
# Export final : mêmes transformations train/test, mêmes features retenues
selected_features = top_corr

train_export = X_train_preprocessed[selected_features].copy()
train_export[target_col] = y_train.to_numpy()

test_export = X_test_preprocessed[selected_features].copy()
test_export[target_col] = y_test.to_numpy()

df_to_save = pd.concat([train_export, test_export], axis=0).sort_index()

df_to_save.to_csv(
    r"C:\Users\tidia\Documents-org\projet5-codingweek\data\processed\features_and_target.csv",
    index=False
 )

data_processed = pd.read_csv(
    r"C:\Users\tidia\Documents-org\projet5-codingweek\data\processed\features_and_target.csv"
 )
print(data_processed.head())
print(f"Train exporté : {train_export.shape} | Test exporté : {test_export.shape}")

In [ ]:
# Prêt pour les modèles
print("EDA complétée : dataset préparé sans fuite de données et sauvegardé dans data/processed/features_and_target.csv")
print(f"Features sélectionnées sur le train uniquement : {len(selected_features)} colonnes")
print(f"Échantillons exportés : {len(df_to_save)}")
print(f"Classes : {df_to_save[target_col].value_counts().to_dict()}")
print(f"Split initial leakage-safe : train={len(X_train_raw)}, test={len(X_test_raw)}")